# 04. 임베딩 (Embedding)

텍스트를 **숫자 벡터**로 변환합니다.  
의미가 비슷한 텍스트는 벡터 공간에서 가까운 위치에 놓입니다.

```
"RAG" → [0.021, -0.053, 0.118, ...]  (1536차원)
"검색 증강 생성" → [0.019, -0.047, 0.121, ...]  (비슷한 벡터)
"사과" → [-0.312, 0.445, -0.088, ...]  (다른 벡터)
```

In [1]:
from dotenv import load_dotenv
load_dotenv(override=True, dotenv_path="../.env")

True

## 1. OpenAIEmbeddings 초기화

In [2]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
print(embeddings)

client=<openai.resources.embeddings.Embeddings object at 0x0000026BA7E468A0> async_client=<openai.resources.embeddings.AsyncEmbeddings object at 0x0000026BAA8C8A10> model='text-embedding-3-small' dimensions=None deployment='text-embedding-ada-002' openai_api_version=None openai_api_base=None openai_api_type=None openai_proxy=None embedding_ctx_length=8191 openai_api_key=SecretStr('**********') openai_organization=None allowed_special=None disallowed_special=None chunk_size=1000 max_retries=2 request_timeout=None headers=None tiktoken_enabled=True tiktoken_model_name=None show_progress_bar=False model_kwargs={} skip_empty=False default_headers=None default_query=None retry_min_seconds=4 retry_max_seconds=20 http_client=None http_async_client=None check_embedding_ctx_length=True


## 2. 단일 텍스트 임베딩 - embed_query()

In [3]:
vector = embeddings.embed_query("RAG란 무엇인가요?")

print(f"벡터 차원: {len(vector)}")
print(f"앞 5개 값: {vector[:5]}")

벡터 차원: 1536
앞 5개 값: [0.020294189453125, 0.018463134765625, -0.034332275390625, 0.0313720703125, -0.007678985595703125]


## 3. 여러 문서 임베딩 - embed_documents()

In [4]:
texts = [
    "RAG는 검색과 생성을 결합한 기술입니다.",
    "LangChain은 LLM 애플리케이션 프레임워크입니다.",
    "벡터 스토어는 임베딩을 저장합니다."
]

vectors = embeddings.embed_documents(texts)

print(f"임베딩한 문서 수: {len(vectors)}")
print(f"각 벡터 차원: {len(vectors[0])}")

임베딩한 문서 수: 3
각 벡터 차원: 1536


## 4. 코사인 유사도로 관련성 측정

코사인 유사도: -1 ~ 1 사이 값, **1에 가까울수록 유사**

In [5]:
import numpy as np

def cosine_similarity(v1, v2):
    v1, v2 = np.array(v1), np.array(v2)
    return np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))

query = "RAG 기술을 알려주세요"
query_vec = embeddings.embed_query(query)

candidates = [
    "RAG는 검색 증강 생성 기술입니다.",
    "LangChain은 LLM 프레임워크입니다.",
    "오늘 점심 메뉴는 뭘 먹을까요?"
]

print(f"쿼리: '{query}'\n")
for text in candidates:
    vec = embeddings.embed_query(text)
    score = cosine_similarity(query_vec, vec)
    print(f"  유사도 {score:.4f} | {text}")

쿼리: 'RAG 기술을 알려주세요'

  유사도 0.6609 | RAG는 검색 증강 생성 기술입니다.
  유사도 0.2977 | LangChain은 LLM 프레임워크입니다.
  유사도 0.1452 | 오늘 점심 메뉴는 뭘 먹을까요?


## 5. embed_query vs embed_documents 차이

| 메서드 | 용도 | 입력 |
|---|---|---|
| `embed_query()` | 사용자 질문 임베딩 | 단일 문자열 |
| `embed_documents()` | 문서 청크 임베딩 | 문자열 리스트 |

In [6]:
# 실제로는 내부적으로 약간 다른 처리를 하지만 결과 벡터 차원은 동일
q_vec = embeddings.embed_query("RAG란?")
d_vec = embeddings.embed_documents(["RAG란?"])[0]

print(f"embed_query 차원:     {len(q_vec)}")
print(f"embed_documents 차원: {len(d_vec)}")

embed_query 차원:     1536
embed_documents 차원: 1536


## 정리

- 임베딩은 텍스트의 **의미**를 숫자 벡터로 표현합니다.
- 유사한 의미 → 유사한 벡터 → 높은 코사인 유사도
- 이 원리를 이용해 관련 문서를 검색합니다.

다음 단계: **벡터 스토어(Vector Store)** →